# Gradient-boosted models for perpetual-funding signals

This execution notebook runs every declared gradient-boosted configuration and every declared
checkpoint through the shared request boundary. Checkpoint identity is retained in the prediction
catalog. Comparative interpretation is deferred to `12_model_analysis`.

**Learning objectives**

- construct a complete gradient-boosting request grid from the published menu;
- inspect fold-scaled loss settings and checkpoint membership before fitting; and
- verify that fitted checkpoints remain distinct catalog identities.

**Book reference:** Chapter 12, gradient boosting for trading.

**Prerequisites:** finalized crypto labels, features, and purged walk-forward folds.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    ALL_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    run_model_plan,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = ALL_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {}

## Resolve the complete grid

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(study) if EXECUTION_TIER == "canonical" else None
)
requests = model_request_catalog("gbm", labels=LABELS)
requests

family,label,config_name
str,str,str
"""gbm""","""fwd_ret_8h""","""default_mse"""
"""gbm""","""fwd_ret_8h""","""default_mae"""
"""gbm""","""fwd_ret_8h""","""default_huber"""
"""gbm""","""fwd_ret_8h""","""leaves_7_mse"""
"""gbm""","""fwd_ret_8h""","""leaves_7_mae"""
…,…,…
"""gbm""","""fwd_dir_8h_3c""","""default_multiclass"""
"""gbm""","""fwd_dir_8h_3c""","""leaves_7_multiclass"""
"""gbm""","""fwd_dir_8h_3c""","""leaves_15_multiclass"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# One planned row per boosting checkpoint; group them back to the configuration that declares them.
checkpoint_contracts = (
    declared_contracts(plan)
    .group_by("label", "config_name", "training_hash", "eligible_rows", maintain_order=True)
    .agg(pl.col("checkpoint_value").alias("checkpoints"))
)
checkpoint_contracts

label,config_name,training_hash,eligible_rows,checkpoints
str,str,str,i64,list[i64]
"""fwd_ret_8h""","""default_mse""","""4758383c4a65""",35280,"[50, 100, … 500]"
"""fwd_ret_8h""","""default_mae""","""373c99c5fef9""",35280,"[50, 100, … 500]"
"""fwd_ret_8h""","""default_huber""","""27cc35c6a25a""",35280,"[50, 100, … 500]"
"""fwd_ret_8h""","""leaves_7_mse""","""f14f18ffdbcd""",35280,"[50, 100, … 500]"
"""fwd_ret_8h""","""leaves_7_mae""","""bc818261ac7d""",35280,"[50, 100, … 500]"
…,…,…,…,…
"""fwd_dir_8h_3c""","""default_multiclass""","""218d0d737d6c""",35280,"[50, 100, … 500]"
"""fwd_dir_8h_3c""","""leaves_7_multiclass""","""37bff8802022""",35280,"[50, 100, … 500]"
"""fwd_dir_8h_3c""","""leaves_15_multiclass""","""896c0e2e5715""",35280,"[50, 100, … 500]"


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## Execute and validate catalog membership

In [6]:
execution = run_model_plan(
    plan,
    population_name="crypto-gbm-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("GBM checkpoint population is incomplete")
catalog.select(
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

      fold 0: training n_train=31,402 n_val=18,542 trees=500 num_leaves=? obj=regression


      fold 0: done in 0s


      fold 0: training n_train=31,402 n_val=18,542 trees=500 num_leaves=? obj=regression_l1


      fold 0: done in 0s


      fold 0: training n_train=31,402 n_val=18,542 trees=500 num_leaves=? obj=huber


...

      fold 1: training n_train=23,681 n_val=16,738 trees=500 num_leaves=31 obj=multiclass


      fold 1: done in 1s


      fold 1: training n_train=23,681 n_val=16,738 trees=500 num_leaves=63 obj=multiclass


      fold 1: done in 2s


label,config_name,checkpoint_kind,checkpoint_value,training_hash,prediction_hash,complete
str,str,str,i64,str,str,bool
"""fwd_dir_8h""","""default_binary""","""iteration""",50,"""5adbe89dca03""","""5ee9323b90a3""",true
"""fwd_dir_8h""","""default_binary""","""iteration""",100,"""5adbe89dca03""","""edbc302e21bb""",true
"""fwd_dir_8h""","""default_binary""","""iteration""",150,"""5adbe89dca03""","""46bbc0407015""",true
"""fwd_dir_8h""","""default_binary""","""iteration""",200,"""5adbe89dca03""","""bac9ef356de4""",true
"""fwd_dir_8h""","""default_binary""","""iteration""",250,"""5adbe89dca03""","""c350b988b181""",true
…,…,…,…,…,…,…
"""fwd_ret_8h""","""leaves_7_mse""","""iteration""",300,"""f14f18ffdbcd""","""10567c68c16c""",true
"""fwd_ret_8h""","""leaves_7_mse""","""iteration""",350,"""f14f18ffdbcd""","""bfca003be162""",true
"""fwd_ret_8h""","""leaves_7_mse""","""iteration""",400,"""f14f18ffdbcd""","""33ab28ea16c2""",true


## Key takeaways and limitations

- A checkpoint is part of a model configuration and remains available for validation backtests.
- Robust-loss thresholds are resolved at the scale of each training fold.
- The notebook establishes execution and lineage, not which model should be traded.